# Clase 114 — Lion y Sophia desde scratch (numpy)

Lion (Google, 2023) = sign-based update, sin momentum de 2do orden.
Sophia (Stanford, 2023) = aproximación diagonal del Hessian.

Comparamos contra Adam manual sobre Rosenbrock.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Rosenbrock: f(x,y) = (1-x)^2 + 100*(y-x^2)^2; mínimo en (1, 1)
def rosen(w):
    x, y = w
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosen_grad(w):
    x, y = w
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

## 1. Adam (baseline)

In [ ]:
def adam(w0, lr=0.005, b1=0.9, b2=0.999, eps=1e-8, steps=200):
    w = w0.copy(); m = np.zeros_like(w); v = np.zeros_like(w)
    traj = [w.copy()]
    for t in range(1, steps+1):
        g = rosen_grad(w)
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*(g**2)
        m_hat = m / (1 - b1**t); v_hat = v / (1 - b2**t)
        w -= lr * m_hat / (np.sqrt(v_hat) + eps)
        traj.append(w.copy())
    return np.array(traj)

traj_adam = adam(np.array([-1.5, 1.5]))
print(f'Adam final: {traj_adam[-1]}, loss={rosen(traj_adam[-1]):.4f}')

## 2. Lion (sign-based)

Update: `c = b1*m + (1-b1)*g; w -= lr*sign(c); m = b2*m + (1-b2)*g`

Notar: la dirección es solo `+1/-1/0` — sin magnitud. Ahorra memoria (no v), funciona con lr ~3-10x menor que Adam.

In [ ]:
def lion(w0, lr=0.001, b1=0.9, b2=0.99, steps=200):
    w = w0.copy(); m = np.zeros_like(w)
    traj = [w.copy()]
    for t in range(steps):
        g = rosen_grad(w)
        c = b1*m + (1-b1)*g
        w -= lr * np.sign(c)
        m = b2*m + (1-b2)*g
        traj.append(w.copy())
    return np.array(traj)

traj_lion = lion(np.array([-1.5, 1.5]))
print(f'Lion final: {traj_lion[-1]}, loss={rosen(traj_lion[-1]):.4f}')

## 3. Sophia (Hessian diagonal)

Usa una estimación numérica de la diagonal del Hessian via `H_ii ≈ (g(w+eps e_i) - g(w))/eps`.
Update: `w -= lr * clip(g / max(H, eps), max_value)`

In [ ]:
def hessian_diag_numeric(w, eps=1e-4):
    g0 = rosen_grad(w)
    H = np.zeros_like(w)
    for i in range(len(w)):
        wp = w.copy(); wp[i] += eps
        H[i] = (rosen_grad(wp)[i] - g0[i]) / eps
    return H

def sophia(w0, lr=0.01, rho=1.0, steps=200, hess_every=5):
    w = w0.copy(); H = np.ones_like(w)
    traj = [w.copy()]
    for t in range(steps):
        g = rosen_grad(w)
        if t % hess_every == 0:
            H = 0.99 * H + 0.01 * np.abs(hessian_diag_numeric(w))
        update = np.clip(g / np.maximum(H, 1e-6), -rho, rho)
        w -= lr * update
        traj.append(w.copy())
    return np.array(traj)

traj_sophia = sophia(np.array([-1.5, 1.5]))
print(f'Sophia final: {traj_sophia[-1]}, loss={rosen(traj_sophia[-1]):.4f}')

## 4. Trayectorias sobre el paisaje

In [ ]:
xs = np.linspace(-2, 2, 200); ys = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(xs, ys); Z = (1-X)**2 + 100*(Y-X**2)**2

fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(X, Y, np.log10(Z+1), levels=30, cmap='gray', alpha=0.5)
ax.plot(*traj_adam.T, 'b.-', label='Adam', alpha=0.8, markersize=3)
ax.plot(*traj_lion.T, 'r.-', label='Lion', alpha=0.8, markersize=3)
ax.plot(*traj_sophia.T, 'g.-', label='Sophia', alpha=0.8, markersize=3)
ax.plot(1, 1, 'k*', markersize=15, label='óptimo')
ax.legend(); ax.set_title('Optimizadores sobre Rosenbrock')
plt.show()

## 5. Wall-clock + convergencia

In [ ]:
import time

results = {}
for name, fn in [('Adam', adam), ('Lion', lion), ('Sophia', sophia)]:
    t0 = time.perf_counter()
    tr = fn(np.array([-1.5, 1.5]))
    elapsed = time.perf_counter() - t0
    losses = np.array([rosen(w) for w in tr])
    results[name] = (elapsed, losses)
    print(f'{name:8s}: {elapsed*1000:.1f} ms, final_loss={losses[-1]:.6f}')

fig, ax = plt.subplots(figsize=(8, 4))
for name, (_, losses) in results.items():
    ax.semilogy(losses, label=name)
ax.legend(); ax.set_xlabel('step'); ax.set_ylabel('loss (log)'); ax.set_title('Convergencia')
plt.show()

## Conclusiones

- **Lion**: 1 buffer (solo m). Memoria ~50% Adam. lr ~10x más chico. Excelente en transformers grandes.
- **Sophia**: estima Hessian → mejor curvatura. Usado en GPT-style pretraining (Stanford 2023, ~2x speedup).
- Ambos no son drop-in para todo modelo: requieren tunear lr/wd. Adam sigue siendo default seguro.

## ✅ Soluciones de los ejercicios

Optimizadores modernos (Lion, Sophia, Schedule-Free) sobre un ViT en CIFAR-10, en PyTorch. Los optimizadores especializados viven en paquetes aparte (`lion-pytorch`, `sophia`, `schedulefree`); las celdas degradan con `try/except` y se validan por AST. El **5** (memoria) es aritmética pura, ejecutable.

**Ej. 1 — AdamW baseline.** ViT en CIFAR-10 con `AdamW(lr=1e-3, wd=0.05)` (config estándar).

In [ ]:
import torch
import torch.nn as nn

# ViT-Tiny de juguete (en la practica: timm.create_model('vit_tiny_patch16_224', num_classes=10))
model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 192), nn.GELU(), nn.Linear(192, 10))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
print("Baseline AdamW: lr=1e-3, wd=0.05. Estado por parametro: m + v (2 buffers).")

**Ej. 2 — Lion.** Misma red, `lr=1e-4, wd=0.5`: LR ~10x menor, weight decay mayor, y un solo buffer de estado.

In [ ]:
import torch
import torch.nn as nn
try:
    from lion_pytorch import Lion            # pip install lion-pytorch
except ImportError:
    Lion = None

model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 192), nn.GELU(), nn.Linear(192, 10))
if Lion is not None:
    opt = Lion(model.parameters(), lr=1e-4, weight_decay=0.5)
    print("Lion: update = sign(interp(grad, momentum)). Un solo buffer (m) -> menos VRAM.")
else:
    print("lion-pytorch no instalado. En Keras: keras.optimizers.Lion(1e-4, weight_decay=0.5)")

**Ej. 3 — Sophia.** Precondiciona con una estimación de la diagonal del Hessiano (Hutchinson) cada 10 steps.

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 192), nn.GELU(), nn.Linear(192, 10))
try:
    from sophia import SophiaG               # pip install sophia-opt (referencia)
    opt = SophiaG(model.parameters(), lr=1e-4, rho=0.04, weight_decay=0.1)
    hess_update_every = 10
    print("Sophia: estima diag(Hessian) via Hutchinson cada", hess_update_every, "steps.")
except ImportError:
    print("sophia-opt no instalado. Idea: curvatura de 2do orden barata -> converge en menos pasos.")

**Ej. 4 — Schedule-Free.** `AdamWScheduleFree(lr=1e-3, warmup_steps=500)`: sin cosine, combina warmup + promediado de iterados.

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 192), nn.GELU(), nn.Linear(192, 10))
try:
    from schedulefree import AdamWScheduleFree   # pip install schedulefree
    opt = AdamWScheduleFree(model.parameters(), lr=1e-3, warmup_steps=500)
    opt.train()      # IMPORTANTE: opt.train() antes de entrenar, opt.eval() antes de evaluar
    print("AdamWScheduleFree: warmup 500 steps y luego LR constante. Sin scheduler externo.")
except ImportError:
    print("schedulefree no instalado. Elimina el tuneo del schedule: solo warmup + promedio de iterados.")

**Ej. 5 — Memoria.** El estado del optimizador = `(buffers/param) × n_params × 4 bytes`. Lion ahorra la mitad vs AdamW.

In [ ]:
import torch
import torch.nn as nn

def optimizer_state_mb(model, buffers_per_param):
    n = sum(p.numel() for p in model.parameters())
    return n * buffers_per_param * 4 / 1e6      # float32 = 4 bytes

model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 768), nn.GELU(), nn.Linear(768, 10))
for name, bpp in [("SGD", 0), ("SGD+momentum", 1), ("Lion", 1), ("AdamW", 2), ("Sophia", 2)]:
    print(f"{name:14s}: estado optimizer ~ {optimizer_state_mb(model, bpp):.2f} MB")
print("Regla: menos buffers/param = menos VRAM. Lion (1) usa la mitad que AdamW (2).")